## SRP556713

**paper:** [PMID: 39894335](https://www.sciencedirect.com/science/article/pii/S0300908425000215?via%3Dihub) - A set of microRNAs are differentially expressed in cachexic naked mole rat colony members after chronic heavy burden under normoxia, 2025

**date, curator:** 2025-09-04, Sara Carsanaro

**notes**
* per methods, "muscle" is skeletal muscle
* for protocol, methods say they use:
    * miRNEasy kits to isolate RNA
    * NEBnext kits to prepare cDNA libraries
    * not enough info to list a specific kit but enough to list RNASelection as miRNA

### annotation summary
run this after annotation is complete

In [27]:
anat_summary = library_to_add[['infoOrgan', 'anatId', 'anatName', 'anatAnnotationStatus']]
unique_anat = anat_summary.drop_duplicates()
display_df(unique_anat)

,infoOrgan,anatId,anatName,anatAnnotationStatus
0,Liver,UBERON:0002107,liver,perfect match
1,Muscle,UBERON:0001134,skeletal muscle tissue,perfect match
4,Heart,UBERON:0000948,heart,perfect match
7,Kidney,UBERON:0002113,kidney,perfect match


In [28]:
dev_summary = library_to_add[['infoStage', 'stageId', 'stageName', 'stageAnnotationStatus']]
unique_dev = dev_summary.drop_duplicates()
display_df(unique_dev)

,infoStage,stageId,stageName,stageAnnotationStatus
0,"5,6 years",UBERON:0000113,post-juvenile adult stage,missing child term
1,"6,2 years",UBERON:0000113,post-juvenile adult stage,missing child term
2,4 year,UBERON:0000113,post-juvenile adult stage,missing child term
4,"6,6 year",UBERON:0000113,post-juvenile adult stage,missing child term
5,"3,6 years",UBERON:0000113,post-juvenile adult stage,missing child term


### set variables, import packages, define functions

In [1]:
experiment_id = "SRP556713"

path_to_create_exp_script = "/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py" 
experiment_type = "bulk"

path_to_output_main = "/Users/scarsana/Desktop/git/expression-annotations/Notebooks/bulk/" 
path_to_output = "{}{}/".format(path_to_output_main, experiment_id)
library_path_from_script = "{}RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_path_from_script = "{}RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
library_to_add_path = "{}complete_RNASeqLibrary_{}.tsv".format(path_to_output, experiment_id)
experiment_to_add_path = "{}complete_RNASeqExperiment_{}.tsv".format(path_to_output, experiment_id)
script_file = "{}.ipynb".format(experiment_id)
commit_message_exp = '"adding annotated bulk experiment {}"'.format(experiment_id)
commit_message_py = '"adding annotation files for {} to notebook folder"'.format(experiment_id)


## to add to git
path_to_git_annotations = "/Users/scarsana/Desktop/git/expression-annotations/RNA_Seq/"
git_library_path = "{}RNASeqLibrary.tsv".format(path_to_git_annotations)
git_experiment_path = "{}RNASeqExperiment.tsv".format(path_to_git_annotations)

library_cols = ['#libraryId', 'experimentId', 'platform', 'SRSId', 'anatId', 'anatName', 'stageId', 'stageName', 'url_GSM', 'infoOrgan', 'infoStage', 'anatAnnotationStatus', 'anatBiologicalStatus', 'stageAnnotationStatus', 'sex', 'strain', 'genotype', 'speciesId', 'protocol', 'protocolType', 'RNASelection', 'globin_reduction', 'replicate', 'lib_name', 'sampleName', 'sampleAge_value', 'sampleAge_unit', 'PATOid', 'PATOname','EFOid', 'EFOname','comment', 'condition', 'physiologicalStatus', 'annotatorId', 'lastModificationDate']

In [2]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
import pandas as pd
import numpy as np
from IPython.display import display, HTML
import os
import csv

# displays df with the scrollbar next to the DataFrame
def display_df(df):
    pd.set_option("display.max_rows", None)
    pd.set_option("display.max_columns", None)
    display(HTML("<div style='height: 300px; overflow: auto; width: fit-content'>" +
        df.style.to_html(index=False) + "</div>"))

# function that compares two columns in a dataframe and tells you which ones are not equal (case insensitive)
def compare_columns(df, col1, col2, return_col):
    compare_return = df[col1].str.lower() != df[col2].str.lower()  
    df.loc[compare_return, return_col] 
    if not any(compare_return):
        print("The two columns are equal (case insensitive)")
    else:
        print("The following rows are not equal: ")
        print(df.loc[compare_return, return_col])

# fixes formatting of file to match libreoffice settings/historic file format
def update_format(path):
    with open(path, 'r') as file:
        filedata = file.read()
    # Replace the target string
    filedata = filedata.replace("\t\"\"", "\t")
    # Write the file out again
    with open(path, 'w') as file:
        file.write(filedata)

# checks for duplicate values in a specific column and prints those values + the corresponding library id
def dup_check(df, column):
    duplicateCheck = df.duplicated(subset=[column], keep=False)
    duplicateCheck.sort_values(inplace=True)
    if duplicateCheck.unique().any() == False:
        print("no duplicate values in " + column)
    elif duplicateCheck.unique().any() == True and column != '#libraryId':
        dups = df[duplicateCheck].loc[:,['#libraryId', column]]
        df_dups = pd.DataFrame(dups)
        df_dups.sort_values(inplace=True, by=column)
        print(df_dups)
    elif duplicateCheck.unique().any() == True and column == '#libraryId':
        print(df[duplicateCheck].loc[:,['#libraryId']])

# prints all unique values in a specific column
def unique_sorted(df, column):
    unique = df[column].unique()
    unique.sort()
    print(unique)

/var/folders/b5/crkp117d43q5mcndnwlrww3w0000gn/T/ipykernel_24273/3311601719.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


### script

In [3]:
! python3 $path_to_create_exp_script $experiment_id $path_to_output $experiment_type

/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:120: SyntaxWarning: invalid escape sequence '\('
  all_protoc = [w.replace('(', '\(') for w in all_protoc]
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:121: SyntaxWarning: invalid escape sequence '\)'
  all_protoc = [w.replace(')', '\)') for w in all_protoc] 
/Users/scarsana/Desktop/git/scRNA-Seq/scripts/Create_ExpLib_tables.py:11: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd
Be patient, it may take a few minutes.
100%|███████████████████████████████████████████| 24/24 [00:23<00:00,  1.01it/s]
0 samples 

### library annnotations

In [4]:
library = pd.read_csv(library_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX27323669,SRP556713,NextSeq 550,SRS23763307,,,,,,Liver,"5,6 years",,,,M,,,10181,,,,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221995,,,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_liver_C1,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968269,8A04F9C1E30769E79F425207F6C9524A
1,SRX27323668,SRP556713,NextSeq 550,SRS23763306,,,,,,Muscle,"6,2 years",,,,M,,,10181,,,,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221998,,,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_muscle_C3,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968270,6F4848DF513A54ACFAE862F7D5D606A9
2,SRX27323667,SRP556713,NextSeq 550,SRS23763305,,,,,,Muscle,4 year,,,,M,,,10181,,,,,,Model organism or animal sample from Heterocephalus glaber,SAMN46014957,,,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chi

#### anatomical entity
* [uberon ols](https://www.ebi.ac.uk/ols4/ontologies/uberon)

In [5]:
unique_sorted(library, "infoOrgan")

['Heart' 'Kidney' 'Liver' 'Muscle']


In [6]:

# Heart
library.loc[library["infoOrgan"] == "Heart", "anatId"] = "UBERON:0000948"
library.loc[library["infoOrgan"] == "Heart", "anatName"] = "heart"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Heart", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Heart", "anatBiologicalStatus"] = "not documented"

# Kidney
library.loc[library["infoOrgan"] == "Kidney", "anatId"] = "UBERON:0002113"
library.loc[library["infoOrgan"] == "Kidney", "anatName"] = "kidney"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Kidney", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Kidney", "anatBiologicalStatus"] = "not documented"

# Liver
library.loc[library["infoOrgan"] == "Liver", "anatId"] = "UBERON:0002107"
library.loc[library["infoOrgan"] == "Liver", "anatName"] = "liver"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Liver", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Liver", "anatBiologicalStatus"] = "not documented"

# Muscle - skeletal muscle per methods
library.loc[library["infoOrgan"] == "Muscle", "anatId"] = "UBERON:0001134"
library.loc[library["infoOrgan"] == "Muscle", "anatName"] = "skeletal muscle tissue"
# perfect match, missing child term, other
library.loc[library["infoOrgan"] == "Muscle", "anatAnnotationStatus"] = "perfect match"
# full sampling, partial sampling, not documented
library.loc[library["infoOrgan"] == "Muscle", "anatBiologicalStatus"] = "not documented"

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX27323669,SRP556713,NextSeq 550,SRS23763307,UBERON:0002107,liver,,,,Liver,"5,6 years",perfect match,not documented,,M,,,10181,,,,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221995,,,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_liver_C1,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968269,8A04F9C1E30769E79F425207F6C9524A
1,SRX27323668,SRP556713,NextSeq 550,SRS23763306,UBERON:0001134,skeletal muscle tissue,,,,Muscle,"6,2 years",perfect match,not documented,,M,,,10181,,,,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221998,,,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_muscle_C3,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968270,6F4848DF513A54ACFAE862F7D5D606A9
2,SRX27323667,SRP556713,NextSeq 550,SRS23763305,UBERON:0001134,skeletal muscle tissue,,,,Muscle,4 year,perfect match,not documented,,M,,,10181,,,,,,Model organism or animal sample from Heterocephalus glaber,SAMN46014957,,,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obt

#### stage
- [species specific developmental ontologies](https://github.com/obophenotype/developmental-stage-ontologies/tree/master/src/ontology/components)

In [7]:
unique_sorted(library, "infoStage")

['3,6 years' '4 year' '5,6 years' '6,2 years' '6,6 year']


In [8]:
# all
library.loc[:,'stageId'] = 'UBERON:0000113'
library.loc[:,'stageName'] = 'post-juvenile adult stage'
# perfect match, missing child term, other
library.loc[:,'stageAnnotationStatus'] = 'missing child term'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX27323669,SRP556713,NextSeq 550,SRS23763307,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,"5,6 years",perfect match,not documented,missing child term,M,,,10181,,,,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221995,,,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_liver_C1,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968269,8A04F9C1E30769E79F425207F6C9524A
1,SRX27323668,SRP556713,NextSeq 550,SRS23763306,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,"6,2 years",perfect match,not documented,missing child term,M,,,10181,,,,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221998,,,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_muscle_C3,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968270,6F4848DF513A54ACFAE862F7D5D606A9
2,SRX27323667,SRP556713,NextSeq 550,SRS23763305,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,4 year,perfect match,not documented,missing child term,M,,,10181,,,,,,Model organism or animal sample from Heterocephalus glaber,SAMN46014957,,,,,,,,,,,04/09/2025,"Liver, kidney, heart, and

#### sex, strain, genotype, speciesId
- uniprot [strain list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/strains)
- uniprot [species list](https://ftp.uniprot.org/pub/databases/uniprot/current_release/knowledgebase/complete/docs/speclist)
- bgee [strain mapping](https://gitlab.sib.swiss/Bgee/expression-annotations/-/tree/develop/Strains?ref_type=heads)

In [ ]:
#library.loc[library["sex"] == "male", "sex"] = "M"
#library.loc[library["sex"] == "female", "sex"] = "F"

#library.loc[:,'strain'] = ''

#library.loc[:,'genotype'] = ''

#library.loc[:,'speciesId'] = ''

# view
display_df(library)

#### protocol
see [bulk kits](https://gitlab.sib.swiss/Bgee/scRNA-Seq/-/blob/main/scripts/bulk_kits.csv) for some common protocols

In [10]:
# making these variables because we use them again in the experiment file
#my_protocol = ''
# full_length or 3'
#my_protocolType = ''

#library.loc[:,'protocol'] = my_protocol
#library.loc[:,'protocolType'] = my_protocolType
# polyA, ribo-minus, miRNA, lncRNA, circRNA
library.loc[:,'RNASelection'] = 'miRNA'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX27323669,SRP556713,NextSeq 550,SRS23763307,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,"5,6 years",perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221995,,,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_liver_C1,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968269,8A04F9C1E30769E79F425207F6C9524A
1,SRX27323668,SRP556713,NextSeq 550,SRS23763306,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,"6,2 years",perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221998,,,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_muscle_C3,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968270,6F4848DF513A54ACFAE862F7D5D606A9
2,SRX27323667,SRP556713,NextSeq 550,SRS23763305,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,4 year,perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,Model organism or animal sample from Heterocephalus glaber,SAMN46014957,,,,,,,,,,,04/09/2025,"Liver, kid

#### globin, replicates

In [15]:
# check for duplicate SRSId values
dup_check(library, "SRSId")

no duplicate values in SRSId


In [ ]:
#library.loc[:,'globin_reduction'] = 'Y'

# replicates
#library.loc[library["#libraryId"] == "old", "replicate"] = "1"
#library.loc[library["#libraryId"].isin(["one", "two"]), "replicate"] = "1"

# view
display_df(library)

In [11]:
unique_sorted(library, "infoStage")

['3,6 years' '4 year' '5,6 years' '6,2 years' '6,6 year']


#### sample age, pato, physiological status
* [PATO](https://www.ebi.ac.uk/ols4/ontologies/pato)
* [EFO](https://www.ebi.ac.uk/ols4/ontologies/efo)

In [12]:
library.loc[library["infoStage"] == "3,6 years", "sampleAge_value"] = "3.6"
library.loc[library["infoStage"] == "4 year", "sampleAge_value"] = "4"
library.loc[library["infoStage"] == "5,6 years", "sampleAge_value"] = "5.6"
library.loc[library["infoStage"] == "6,2 years", "sampleAge_value"] = "6.2"
library.loc[library["infoStage"] == "6,6 year", "sampleAge_value"] = "6.6"
library.loc[:,'sampleAge_unit'] = 'year'

# ex. castrated male
#library.loc[:,'PATOid'] = ''
#library.loc[:,'PATOname'] = ''

# ex. castrated, pregnant, pre-smoltification, post-smoltification, laying eggs
#library.loc[:,'physiologicalStatus'] = ''

# ex. left, right
#library.loc[:,'EFOid'] = ''
#library.loc[:,'EFOname'] = ''

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX27323669,SRP556713,NextSeq 550,SRS23763307,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,"5,6 years",perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221995,5.6,year,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_liver_C1,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968269,8A04F9C1E30769E79F425207F6C9524A
1,SRX27323668,SRP556713,NextSeq 550,SRS23763306,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,"6,2 years",perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221998,6.2,year,,,,,,,,,04/09/2025,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_muscle_C3,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968270,6F4848DF513A54ACFAE862F7D5D606A9
2,SRX27323667,SRP556713,NextSeq 550,SRS23763305,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,4 year,perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,Model organism or animal sample from Heterocephalus glaber,SAMN46014957,4,year,,,,,,,,,04/

#### condition

In [ ]:
# ex. control, diet, light, reproductive capacity, time post mortem, time post feeding, 
# exercise details, menstruation, personality, litter size 
#library.loc[library["condition"] == "old", "condition"] = "new"

# view
display_df(library)

#### annotator id, last modification date

In [13]:
library.loc[:,'annotatorId'] = 'SAC'
library.loc[:,'lastModificationDate'] = '2025-09-04'

# view
display_df(library)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate,library_contruction_protocol,source_qc,lib_name_2,lib_name_3,source_name,individual,infoStage_2,infoStage_3,Library_Source,Library_Selection,RunIds,ReadHashes
0,SRX27323669,SRP556713,NextSeq 550,SRS23763307,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,"5,6 years",perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221995,5.6,year,,,,,,,,SAC,2025-09-04,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_liver_C1,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968269,8A04F9C1E30769E79F425207F6C9524A
1,SRX27323668,SRP556713,NextSeq 550,SRS23763306,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,"6,2 years",perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,Model organism or animal sample from Heterocephalus glaber,SAMN46221998,6.2,year,,,,,,,,SAC,2025-09-04,"Liver, kidney, heart, and skeletal muscle samples were taken from each animal. Containing microRNA RNA fractions were isolated from tissue samples using commercial miRNEasy kits (Qiagen, USA). Obtained samples were used to create cDNA libraries and subsequent sequencing. The quality of the isolated fractions was assessed using microelectrophoresis on Bioanalyzer chips (Agilent, USA). Samples with an RNA integrity index RIN of at least 8 were selected for sequencing. Sequencing was performed on the NextSeq platform (Illumina, USA) using NextSeq 500/550 High Output v2 kit (Illumina, USA). cDNA libraries from isolated RNA samples were prepared using NEBnext kits (NEB, USA), according to the manual. Qualitative and quantitative analysis of libraries was performed using Bioanalyzer microelectrophoresis (Agilent, USA) and Qubit fluorometry (ThermoFisher, USA). The quality of sequencing was assessed using the BaseSpace service (Illumina, USA) by the following parameters: cluster density, signal intensity in the detection channels, the proportion of clusters that passed the filter, by the output of aligned reads. All parameters were within the acceptable values.",,NMR_muscle_C3,,,,,,TRANSCRIPTOMIC,cDNA,SRR31968270,6F4848DF513A54ACFAE862F7D5D606A9
2,SRX27323667,SRP556713,NextSeq 550,SRS23763305,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,4 year,perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,Model organism or animal sample from Heterocephalus glaber,SAMN46014957,4,year,,,,,,

#### comments

In [ ]:
#library.loc[:,'comment'] = ''

In [18]:
library.loc[:,'lib_name'] = library['lib_name_2'].values

#### save complete file with correct columns

In [19]:
library_file_complete = library[library_cols]
library_file_complete.to_csv(library_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

# view
display_df(library_file_complete)

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
0,SRX27323669,SRP556713,NextSeq 550,SRS23763307,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,"5,6 years",perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,NMR_liver_C1,SAMN46221995,5.6,year,,,,,,,,SAC,2025-09-04
1,SRX27323668,SRP556713,NextSeq 550,SRS23763306,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,"6,2 years",perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,NMR_muscle_C3,SAMN46221998,6.2,year,,,,,,,,SAC,2025-09-04
2,SRX27323667,SRP556713,NextSeq 550,SRS23763305,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,4 year,perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,NMR_muscle_C2,SAMN46014957,4,year,,,,,,,,SAC,2025-09-04
3,SRX27323666,SRP556713,NextSeq 550,SRS23763303,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,4 year,perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,NMR_muscle_C1,SAMN46014956,4,year,,,,,,,,SAC,2025-09-04
4,SRX27323661,SRP556713,NextSeq 550,SRS23763299,UBERON:0000948,heart,UBERON:0000113,post-juvenile adult stage,,Heart,"6,6 year",perfect match,not documented,missing child term,F,,,10181,,,miRNA,,,NMR_heart_C3,SAMN46014952,6.6,year,,,,,,,,SAC,2025-09-04
5,SRX27323660,SRP556713,NextSeq 550,SRS23763298,UBERON:0000948,heart,UBERON:0000113,post-juvenile adult stage,,Heart,"3,6 years",perfect match,not documented,missing child term,F,,,10181,,,miRNA,,,NMR_heart_C2,SAMN46221997,3.6,year,,,,,,,,SAC,2025-09-04
6,SRX27323659,SRP556713,NextSeq 550,SRS23763297,UBERON:0000948,heart,UBERON:0000113,post-juvenile adult stage,,Heart,4 year,perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,NMR_heart_C1,SAMN46014951,4,year,,,,,,,,SAC,2025-09-04
7,SRX27323655,SRP556713,NextSeq 550,SRS23763293,UBERON:0002113,kidney,UBERON:0000113,post-juvenile adult stage,,Kidney,"6,6 year",perfect match,not documented,missing child term,F,,,10181,,,miRNA,,,NMR_kidney_C3,SAMN46014947,6.6,year,,,,,,,,SAC,2025-09-04
8,SRX27323654,SRP556713,NextSeq 550,SRS23763292,UBERON:0002113,kidney,UBERON:0000113,post-juvenile adult stage,,Kidney,4 year,perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,NMR_kidney_C2,SAMN46014946,4,year,,,,,,,,SAC,2025-09-04
9,SRX27323653,SRP556713,NextSeq 550,SRS23763291,UBERON:0002113,kidney,UBERON:0000113,post-juvenile adult stage,,Kidney,4 year,perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,NMR_kidney_C1,SAMN46014945,4,year,,,,,,,,SAC,2025-09-04


### experiment annotations

In [20]:
experiment = pd.read_csv(experiment_path_from_script, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP556713,Differential expression of microRNAs between health and cachectic naked mole rat,"Enrichment of the habitat of captive rodents Heterocephalus glaber (naked mole rats) to implement their innate behavioral pattern of digging dense soil in search of food, paradoxically led to the appearance of unusual animals in the colony. They showed signs of cachexia, distinguished from other animals by a lower temperature (from 31 to 26 degree) and body mass index with decreasing proportion of subcutaneous fat. This animal demonstrated aggressive feeding behavior, but didnt gain weight even after finishing of experiment with intensive physical activity.To clarify the pathogenetic mechanism of the observed phenomenon, microRNA was extracted from the animal tissues and sequenced. Then bioinformatics analysis differential expression of microRNAs between group of health animals and animals with idiopatic cachexia was performed.",SRA,,,,,,,PRJNA1204180,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### experiment and protocol details

In [21]:
# this will give you the number of rows in the complete library file 
# this should be the number of annotated libraries
ann_lib = len(library_file_complete.index)
len(library_file_complete.index)

12

In [22]:
# partial or total
experiment.loc[:,'experimentStatus'] = 'partial'
#experiment.loc[:,'projectTags'] = '' 
# see above cell, also can add as free text
experiment.loc[:,'numberOfAnnotatedLibraries'] = ann_lib

# these variables should already exist from above but if not can just add as free text
#experiment.loc[:,'protocol'] = my_protocol
#experiment.loc[:,'protocolType'] = my_protocolType

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP556713,Differential expression of microRNAs between health and cachectic naked mole rat,"Enrichment of the habitat of captive rodents Heterocephalus glaber (naked mole rats) to implement their innate behavioral pattern of digging dense soil in search of food, paradoxically led to the appearance of unusual animals in the colony. They showed signs of cachexia, distinguished from other animals by a lower temperature (from 31 to 26 degree) and body mass index with decreasing proportion of subcutaneous fat. This animal demonstrated aggressive feeding behavior, but didnt gain weight even after finishing of experiment with intensive physical activity.To clarify the pathogenetic mechanism of the observed phenomenon, microRNA was extracted from the animal tissues and sequenced. Then bioinformatics analysis differential expression of microRNAs between group of health animals and animals with idiopatic cachexia was performed.",SRA,partial,,12,,,,PRJNA1204180,,,"E,r,r,o,r,:, ,U,n,a,b,l,e, ,t,o, ,r,e,t,r,i,e,v,e, ,d,a,t,a,,, ,S,t,a,t,u,s, ,c,o,d,e, ,4,0,4",,


#### paper and xrefs

In [23]:
#experiment.loc[:,'GSE'] = ''
#experiment.loc[:,'Bioproject'] = '' 
experiment.loc[:,'PMID'] = '39894335'
experiment.loc[:,'reference_url'] = 'https://linkinghub.elsevier.com/retrieve/pii/S0300908425000215'
experiment.loc[:,'DOI'] = '10.1016/j.biochi.2025.01.010'
#experiment.loc[:,'xrefs'] = ''

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP556713,Differential expression of microRNAs between health and cachectic naked mole rat,"Enrichment of the habitat of captive rodents Heterocephalus glaber (naked mole rats) to implement their innate behavioral pattern of digging dense soil in search of food, paradoxically led to the appearance of unusual animals in the colony. They showed signs of cachexia, distinguished from other animals by a lower temperature (from 31 to 26 degree) and body mass index with decreasing proportion of subcutaneous fat. This animal demonstrated aggressive feeding behavior, but didnt gain weight even after finishing of experiment with intensive physical activity.To clarify the pathogenetic mechanism of the observed phenomenon, microRNA was extracted from the animal tissues and sequenced. Then bioinformatics analysis differential expression of microRNAs between group of health animals and animals with idiopatic cachexia was performed.",SRA,partial,,12,,,,PRJNA1204180,39894335,https://linkinghub.elsevier.com/retrieve/pii/S0300908425000215,10.1016/j.biochi.2025.01.010,,


#### comments

In [24]:
experiment.loc[:,'comment'] = 'methods say they used NEBnext kits to prepare cDNA libraries and miRNEasy kits to isolate RNA, not enough info to specify a kit name but enough to list RNASelection as miRNA'

display_df(experiment)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
0,SRP556713,Differential expression of microRNAs between health and cachectic naked mole rat,"Enrichment of the habitat of captive rodents Heterocephalus glaber (naked mole rats) to implement their innate behavioral pattern of digging dense soil in search of food, paradoxically led to the appearance of unusual animals in the colony. They showed signs of cachexia, distinguished from other animals by a lower temperature (from 31 to 26 degree) and body mass index with decreasing proportion of subcutaneous fat. This animal demonstrated aggressive feeding behavior, but didnt gain weight even after finishing of experiment with intensive physical activity.To clarify the pathogenetic mechanism of the observed phenomenon, microRNA was extracted from the animal tissues and sequenced. Then bioinformatics analysis differential expression of microRNAs between group of health animals and animals with idiopatic cachexia was performed.",SRA,partial,,12,,,,PRJNA1204180,39894335,https://linkinghub.elsevier.com/retrieve/pii/S0300908425000215,10.1016/j.biochi.2025.01.010,,"methods say they used NEBnext kits to prepare cDNA libraries and miRNEasy kits to isolate RNA, not enough info to specify a kit name but enough to list RNASelection as miRNA"


#### save complete file

In [25]:
experiment.to_csv(experiment_to_add_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)

### QA time

In [26]:
library_to_add = pd.read_csv(library_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
experiment_to_add = pd.read_csv(experiment_to_add_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

#### to add things here

#### check columns match

In [29]:
# pull from git and pull in library/experiment file
! git pull
git_library = pd.read_csv(git_library_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)
git_experiment = pd.read_csv(git_experiment_path, sep='\t', index_col=False, keep_default_na=False, na_values=['NULL','null', 'nan','NaN'], dtype=object)

# library file
if set(library_to_add.columns) == set(git_library.columns):
    print('The columns in the library file match')
else:
    print('The columns in the library file DO NOT MATCH')

# experiment file
if set(experiment_to_add.columns) == set(git_experiment.columns):
    print('The columns in the experiment file match')
else:
    print('The columns in the experiment file DO NOT MATCH')


# maybe to make this something more like "COLUMNS GOOD - LIBRARY" and "COLUMNS BAD - EXPERIMENT"

Already up to date.
The columns in the library file match
The columns in the experiment file match


#### view files

In [30]:
library_git_plus_new = pd.concat([git_library, library_to_add], ignore_index = True, sort = False)
old_length = git_library.shape[0]
start = old_length - 2
end = old_length + 5
view_lib = library_git_plus_new.iloc[start:end]
view_lib

,#libraryId,experimentId,platform,SRSId,anatId,anatName,stageId,stageName,url_GSM,infoOrgan,infoStage,anatAnnotationStatus,anatBiologicalStatus,stageAnnotationStatus,sex,strain,genotype,speciesId,protocol,protocolType,RNASelection,globin_reduction,replicate,lib_name,sampleName,sampleAge_value,sampleAge_unit,PATOid,PATOname,EFOid,EFOname,comment,condition,physiologicalStatus,annotatorId,lastModificationDate
49649,SRX28459319,SRP579692,Illumina NovaSeq 6000,SRS24780674,UBERON:0000305,amnion,EcabDv:0000002,prenatal stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Amniotic membrane,approx 288 days of gestation,perfect match,not documented,missing child term,NA,,,9796,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,L53_control,SAMN48055198,288,gestational day,,,,,annotated using embryo info since amniotic mem...,,,SAC,2025-09-04
49650,SRX28459318,SRP579692,Illumina NovaSeq 6000,SRS24780672,UBERON:0000305,amnion,EcabDv:0000002,prenatal stage,https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi...,Amniotic membrane,approx 288 days of gestation,perfect match,not documented,missing child term,NA,,,9796,NEBNext Ultra RNA Library Prep Kit,full_length,polyA,,,K56_control,SAMN48055199,288,gestational day,,,,,annotated using embryo info since amniotic mem...,,,SAC,2025-09-04
49651,SRX27323669,SRP556713,NextSeq 550,SRS23763307,UBERON:0002107,liver,UBERON:0000113,post-juvenile adult stage,,Liver,"5,6 years",perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,NMR_liver_C1,SAMN46221995,5.6,year,,,,,,,,SAC,2025-09-04
49652,SRX27323668,SRP556713,NextSeq 550,SRS23763306,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,"6,2 years",perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,NMR_muscle_C3,SAMN46221998,6.2,year,,,,,,,,SAC,2025-09-04
49653,SRX27323667,SRP556713,NextSeq 550,SRS23763305,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,4 year,perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,NMR_muscle_C2,SAMN46014957,4,year,,,,,,,,SAC,2025-09-04
49654,SRX27323666,SRP556713,NextSeq 550,SRS23763303,UBERON:0001134,skeletal muscle tissue,UBERON:0000113,post-juvenile adult stage,,Muscle,4 year,perfect match,not documented,missing child term,M,,,10181,,,miRNA,,,NMR_muscle_C1,SAMN46014956,4,year,,,,,,,,SAC,2025-09-04
49655,SRX27323661,SRP556713,NextSeq 550,SRS23763299,UBERON:0000948,heart,UBERON:0000113,post-juvenile adult stage,,Heart,"6,6 year",perfect match,not documented,missing child term,F,,,10181,,,miRNA,,,NMR_heart_C3,SAMN46014952,6.6,year,,,,,,,,SAC,2025-09-04


In [31]:
experiment_git_plus_new = pd.concat([git_experiment, experiment_to_add], ignore_index = True, sort = False)
experiment_git_plus_new.tail(n=3)

,#experimentId,experimentName,experimentDescription,experimentSource,experimentStatus,projectTags,numberOfAnnotatedLibraries,protocol,protocolType,GSE,Bioproject,PMID,reference_url,DOI,xrefs,comment
923,SRP392258,Transcriptome atlas of 34 goose tissues,The studies of goose gene expression and evolu...,SRA,total,,102,,,,PRJNA868424,39874782,https://pmc.ncbi.nlm.nih.gov/articles/PMC11810...,10.1016/j.psj.2025.104842,,did not mark libraries with same SRSid as repl...
924,SRP579692,Decoding the Amniotic Membrane Transcriptome d...,The equine amniotic membrane is a critical bar...,SRA,partial,,6,NEBNext Ultra RNA Library Prep Kit,full_length,GSE295055,PRJNA1252718,40841585,https://pmc.ncbi.nlm.nih.gov/articles/PMC12371...,10.1038/s41598-025-16671-5,,
925,SRP556713,Differential expression of microRNAs between h...,Enrichment of the habitat of captive rodents H...,SRA,partial,,12,,,,PRJNA1204180,39894335,https://linkinghub.elsevier.com/retrieve/pii/S...,10.1016/j.biochi.2025.01.010,,methods say they used NEBnext kits to prepare ...


### add annotations to git

In [32]:
! git pull

Already up to date.


In [33]:
library_git_plus_new.to_csv(git_library_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
experiment_git_plus_new.to_csv(git_experiment_path, sep="\t", index=False, quoting=csv.QUOTE_ALL)
update_format(git_library_path)
update_format(git_experiment_path)

In [34]:
! git status

On branch develop
Your branch is up to date with 'origin/develop'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ../../../RNA_Seq/RNASeqExperiment.tsv
	modified:   ../../../RNA_Seq/RNASeqLibrary.tsv

Untracked files:
  (use "git add <file>..." to include in what will be committed)
	./

no changes added to commit (use "git add" and/or "git commit -a")


In [35]:
! git add $git_experiment_path $git_library_path

In [36]:
! git commit -m $commit_message_exp

[develop ede0259] adding annotated bulk experiment SRP556713
 2 files changed, 13 insertions(+)


In [37]:
! git push

Enumerating objects: 9, done.
Counting objects: 100% (9/9), done.
Delta compression using up to 12 threads
Compressing objects: 100% (5/5), done.
Writing objects: 100% (5/5), 2.46 KiB | 840.00 KiB/s, done.
Total 5 (delta 4), reused 0 (delta 0), pack-reused 0
remote: 
remote: To create a merge request for develop, visit:
remote:   https://gitlab.sib.swiss/Bgee/expression-annotations/-/merge_requests/new?merge_request%5Bsource_branch%5D=develop
remote: 
To https://gitlab.sib.swiss/Bgee/expression-annotations.git
   ea62cf8..ede0259  develop -> develop


### add annotation folder and script to git

In [ ]:
! git status

1. run first two cells (annotation summary)
2. export as html

In [ ]:
! git add $path_to_output

In [ ]:
! git commit -m $commit_message_py

In [ ]:
! git push